<a href="https://colab.research.google.com/github/barkaPiy/PSNN-FS/blob/main/PSNN_FS.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
%config InlineBackend.figure_format = 'retina'
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
np.set_printoptions(precision=3)
import seaborn as sns
import matplotlib.pyplot as plt
%matplotlib inline
np.set_printoptions(suppress=True)

In [ ]:
from google.colab import files
uploaded = files.upload()

Saving cm1.csv to cm1.csv


In [ ]:
import io
dst = pd.read_csv(io.BytesIO(uploaded['cm1.csv']))

In [ ]:
dst.head()

,id,loc,v(g),ev(g),iv(g),n,v,l,d,i,...,lOCode,lOComment,lOBlank,locCodeAndComment,uniq_Op,uniq_Opnd,total_Op,total_Opnd,branchCount,defects
0,1,1.1,1.4,1.4,1.4,1.3,1.30,1.30,1.30,1.30,...,2,2,2,2,1.2,1.2,1.2,1.2,1.4,False
1,2,1.0,1.0,1.0,1.0,1.0,1.00,1.00,1.00,1.00,...,1,1,1,1,1.0,1.0,1.0,1.0,1.0,True
2,3,24.0,5.0,1.0,3.0,63.0,309.13,0.11,9.50,32.54,...,1,0,6,0,15.0,15.0,44.0,19.0,9.0,False
3,4,20.0,4.0,4.0,2.0,47.0,215.49,0.06,16.00,13.47,...,0,0,3,0,16.0,8.0,31.0,16.0,7.0,False
4,5,24.0,6.0,6.0,2.0,72.0,346.13,0.06,17.33,19.97,...,0,0,3,0,16.0,12.0,46.0,26.0,11.0,False


In [ ]:
# -*- coding: utf-8 -*-
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import io
from google.colab import files
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, accuracy_score
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import LabelEncoder, MinMaxScaler

# --- Helper Functions ---
def correlation(dataset, threshold):
    col_corr = set()
    corr_matrix = dataset.corr()
    for i in range(len(corr_matrix.columns)):
        for j in range(i):
            if abs(corr_matrix.iloc[i, j]) > threshold:
                colname = corr_matrix.columns[i]
                col_corr.add(colname)
    return col_corr

def sigmoid(x):
    return 1 / (1 + np.exp(-x))

def evaluate_accuracy(data, labels, W_2, b_2, numFeatures, datalen):
    num_correct = 0
    predVector = np.zeros(shape=(len(data),))

    for i, (d, l) in enumerate(zip(data, labels)):
        a_1 = d.reshape(numFeatures, 1)
        # Linear Summation (Sigma layer)
        z2 = (np.dot(W_2, a_1) + b_2)

        # Product unit (Pi layer) and Activation
        a_3 = sigmoid(np.prod(z2))
        prediction = a_3
        predVector[i] = np.rint(prediction)

        if (prediction >= .5 and l == 1) or (prediction < .5 and l == 0):
            num_correct += 1
    return num_correct / datalen, predVector

# --- Data Loading ---
labels_full = dst['defects']
le = LabelEncoder()
labels_encoded = le.fit_transform(labels_full)
X_full = dst.drop("defects", axis=1)
X_full = dst.drop("id", axis=1)
y_full = labels_encoded.astype(int)

# --- 5-Fold Cross Validation Setup ---
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=0)
fold_results = []

print("Starting 5-Fold Cross Validation (Pure PSNN Implementation)...")

for fold, (train_index, test_index) in enumerate(skf.split(X_full, y_full)):
    # Split the data
    X_train_cv, X_test_cv = X_full.iloc[train_index], X_full.iloc[test_index]
    y_train_cv, y_test_cv = y_full[train_index], y_full[test_index]

    # 1. Feature Selection
    corr_features = correlation(X_train_cv, 0.65)
    X_train_cv = X_train_cv.drop(corr_features, axis=1)
    X_test_cv = X_test_cv.drop(corr_features, axis=1)

    print("Features kept:", X_train_cv.columns.tolist())

    # 2. Imputation
    imputer = SimpleImputer(strategy='median')
    X_train_imputed = imputer.fit_transform(X_train_cv)
    X_test_imputed = imputer.transform(X_test_cv)

    # 3. MinMaxScaler
    scaler = MinMaxScaler()
    X_train_scaled = scaler.fit_transform(X_train_imputed)
    X_test_scaled = scaler.transform(X_test_imputed)

    X_train_sm = X_train_scaled
    y_train_sm = y_train_cv

    # 4. PSNN Model Initialization
    numSamples, numFeatures = X_train_sm.shape
    datalen_train = len(X_train_sm)

    # He Initialization
    W_2 = np.random.randn(2, numFeatures) * np.sqrt(2 / numFeatures)
    b_2 = np.zeros((2, 1))

    step_size = 0.05

    # 5. Training Loop
    for epoch in range(1001):
        ddb_2 = np.zeros(b_2.shape)
        ddW_2 = np.zeros(W_2.shape)

        for d, l in zip(X_train_sm, y_train_sm):
            dpoint = d.reshape(numFeatures, 1)

            # Forward Pass (Sigma -> Pi -> Sigmoid)
            z2 = np.dot(W_2, dpoint) + b_2
            a_3 = sigmoid(np.prod(z2))

            # Backward Pass
            delta_3 = 2 * (a_3 - l) * a_3 * (1 - a_3)

            # Gradient of product unit: partial derivative of prod(z2) wrt each z2_i
            z2_f = z2.flatten()
            grad_prod = np.array([np.prod(np.delete(z2_f, i)) for i in range(len(z2_f))]).reshape(2, 1)

            # Delta for the sigma layer
            dz2 = delta_3 * grad_prod

            ddb_2 += dz2
            ddW_2 += np.dot(dz2, dpoint.T)

        # Update Parameters
        b_2 -= (ddb_2 / datalen_train) * step_size
        W_2 -= (ddW_2 / datalen_train) * step_size

    # 6. Evaluation on Test Fold
    datalen_test = len(X_test_scaled)
    acc, ypred = evaluate_accuracy(X_test_scaled, y_test_cv, W_2, b_2, numFeatures, datalen_test)
    fold_results.append(acc)

    print(f"\n--- Results for Fold {fold+1} ---")
    print(f"Accuracy Score: {acc:.4f}")
    print("Confusion Matrix:")
    print(confusion_matrix(y_test_cv, ypred))
    print("Classification Report:")
    print(classification_report(y_test_cv, ypred))
    print("-" * 30)

print(f"\nAverage 5-Fold Accuracy: {np.mean(fold_results):.4f}")

Starting 5-Fold Cross Validation (Pure PSNN Implementation)...
Features kept: ['loc', 'l', 'locCodeAndComment', 'defects']

--- Results for Fold 1 ---
Accuracy Score: 1.0000
Confusion Matrix:
[[90  0]
 [ 0 10]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        90
           1       1.00      1.00      1.00        10

    accuracy                           1.00       100
   macro avg       1.00      1.00      1.00       100
weighted avg       1.00      1.00      1.00       100

------------------------------
Features kept: ['loc', 'l', 'locCodeAndComment', 'defects']

--- Results for Fold 2 ---
Accuracy Score: 1.0000
Confusion Matrix:
[[90  0]
 [ 0 10]]
Classification Report:
              precision    recall  f1-score   support

           0       1.00      1.00      1.00        90
           1       1.00      1.00      1.00        10

    accuracy                           1.00       100
   macro avg       